# Финальный инференс: ориентация текста 0° / 180°

Этот блокнот воспроизводит финальный `submission.csv`.

Для каждого текстового кропа модель возвращает `p_180` - вероятность того, что изображение перевёрнуто на 180°. Обучения здесь нет: используются готовые веса PP-LCNet x0.25 и зафиксированный финальный pipeline.


## Как устроен финальный инференс

Для каждого изображения модель получает две версии:

- исходный кроп `x`;
- тот же кроп после поворота на 180°.

Ответы объединяются через антисимметризацию логитов:

\[
z_{sym}=\frac{z(x)-z(R_{180}(x))}{2}
\]

После этого применяется temperature scaling:

\[
p_{180}=\sigma(z_{sym}/T), \qquad T=0.8572522561367453
\]

Финальная конфигурация: PP-LCNet x0.25, RGB `160×80`, direct Paddle Inference FP32; OCR и INT8 отключены.


## Настройка путей и загрузка модели

По умолчанию notebook ищет:

```text
./test/images
./sample_submission.csv
```

Можно задать переменные окружения `AVITO_TEST_DIR` и `AVITO_SAMPLE_SUBMISSION`.

Перед загрузкой модели проверяется SHA256 файла весов - это проверка, что используется именно финальная версия модели.


In [ ]:
from pathlib import Path
import csv
import hashlib
import os
import sys

import numpy as np

CWD = Path.cwd().resolve()
REPO_ROOT = CWD if (CWD / "weights").is_dir() else (CWD / "AVITO_BOOTCAMP_REPO").resolve()

if not (REPO_ROOT / "weights").is_dir():
    raise FileNotFoundError("Запустите notebook из корня AVITO_BOOTCAMP_REPO")

TEST_DIR = Path(
    os.environ.get("AVITO_TEST_DIR", REPO_ROOT / "test" / "images")
).expanduser().resolve()

SAMPLE_PATH = Path(
    os.environ.get("AVITO_SAMPLE_SUBMISSION", REPO_ROOT / "sample_submission.csv")
).expanduser().resolve()

OUTPUT_PATH = REPO_ROOT / "outputs" / "submission.csv"
WEIGHTS_PATH = REPO_ROOT / "weights" / "inference.pdiparams"

EXPECTED_WEIGHTS_SHA256 = "08ee1ce4bcdb30dd4e784862334d1df49d80a600294d75daae4ca4c70bc860e9"
EXPECTED_SUBMISSION_SHA256 = "2a7f5c375454a1193a872a53d600214e5cb79df9c1a9906191b7d1c2b1121c86"

PIPELINE = {
    "architecture": "PP-LCNet_x0_25_textline_ori",
    "model": "official PP-LCNet x0.25",
    "weights_sha256": EXPECTED_WEIGHTS_SHA256,
    "preprocessing": "native ResizeImage 160x80 + official normalization",
    "input_shape": [3, 80, 160],
    "resize": {"width": 160, "height": 80, "interpolation": "cv2 INTER_LINEAR"},
    "aspect_handling": "native fixed resize to 160x80 (non-aspect-preserving)",
    "normalization": {
        "scale": 1 / 255,
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225],
        "layout": "CHW",
        "color": "RGB",
    },
    "symmetry": "logit_antisymmetrization",
    "temperature": 0.8572522561367453,
    "backend": "paddle_infer_fp32",
    "cpu_threads": 1,
    "onednn": False,
    "ocr_enabled": False,
    "ocr_blend_alpha": 0.0,
    "int8_enabled": False,
}

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

actual_weights_sha256 = sha256_file(WEIGHTS_PATH)
if actual_weights_sha256 != EXPECTED_WEIGHTS_SHA256:
    raise RuntimeError("SHA256 весов не совпадает с финальной версией модели")

sys.path.insert(0, str(REPO_ROOT))
from src.final_runtime import FinalOrientationRuntime

runtime = FinalOrientationRuntime(REPO_ROOT, PIPELINE)

print(f"Модель загружена: {PIPELINE['model']}")
print(f"SHA256 весов: {actual_weights_sha256}")
print(f"Test: {TEST_DIR}")
print(f"Sample submission: {SAMPLE_PATH}")


## Инференс и создание submission.csv

Порядок изображений берётся из `sample_submission.csv`.


In [ ]:
if not TEST_DIR.is_dir() or not SAMPLE_PATH.is_file():
    print(
        "Конкурсные данные не найдены.\n"
        "Поместите изображения в ./test/images и sample_submission.csv в корень репозитория\n"
        "или задайте AVITO_TEST_DIR и AVITO_SAMPLE_SUBMISSION."
    )
else:
    with SAMPLE_PATH.open(newline="", encoding="utf-8") as stream:
        reader = csv.DictReader(stream)
        if not reader.fieldnames or "image_id" not in reader.fieldnames:
            raise ValueError("sample_submission.csv должен содержать колонку image_id")
        sample_rows = list(reader)

    image_ids = [row["image_id"] for row in sample_rows]
    image_paths = []

    for image_id in image_ids:
        candidates = [TEST_DIR / image_id, TEST_DIR / f"{image_id}.png"]
        image_path = next((p for p in candidates if p.is_file()), None)
        if image_path is None:
            raise FileNotFoundError(f"Не найдено изображение для image_id={image_id!r}")
        image_paths.append(image_path)

    probabilities = runtime.predict_paths(image_paths, batch_size=64)
    probabilities = np.asarray(probabilities, dtype=np.float64)

    if probabilities.shape != (len(image_ids),):
        raise RuntimeError("Количество предсказаний не совпадает с количеством image_id")
    if not np.isfinite(probabilities).all():
        raise RuntimeError("В предсказаниях есть NaN или inf")
    if not ((probabilities >= 0) & (probabilities <= 1)).all():
        raise RuntimeError("Вероятности должны находиться в диапазоне [0, 1]")

    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with OUTPUT_PATH.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.writer(stream)
        writer.writerow(["image_id", "p_180"])
        writer.writerows(zip(image_ids, probabilities))

    actual_submission_sha256 = sha256_file(OUTPUT_PATH)

    print(f"Готово: {OUTPUT_PATH}")
    print(f"Строк: {len(image_ids)}")
    print(f"SHA256: {actual_submission_sha256}")

    if actual_submission_sha256 == EXPECTED_SUBMISSION_SHA256:
        print("Финальный конкурсный submission воспроизведён побайтно.")
    else:
        print(
            "SHA256 отличается от исходного конкурсного submission. "
            "Для другого входного набора это ожидаемо."
        )
